# 🏋️ Notebook 04 — Fine-Tuning the YOLO Detector

**CO543/CO5430 — Traffic Sign Detection**

**Goal**: Fine-tune YOLOv8n on GTSDB using transfer learning from COCO-pretrained weights.

**M3 Deliverable**: Working end-to-end trained detector with preliminary mAP numbers + at least one ablation started.

> 💡 **Tip**: Run this notebook on Google Colab or Kaggle with a free GPU for faster training.  
> Nano (n) or Small (s) model variants are recommended to stay within free-tier limits.

In [1]:
import sys, os
from pathlib import Path

import torch
import yaml
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# ── Check GPU ──────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — training will be slow. Consider using Colab/Kaggle.')

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'gtsdb_yolov8n.yaml'
print(f'Config  : {CONFIG_PATH}')
print(f'Exists  : {CONFIG_PATH.exists()}')

Device  : cpu
⚠️  No GPU found — training will be slow. Consider using Colab/Kaggle.
Config  : E:\traffic-sign-detection\configs\gtsdb_yolov8n.yaml
Exists  : True


## 1. Review Training Configuration

In [2]:
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('── Training Configuration ──')
for k, v in cfg.items():
    print(f'  {k:20s}: {v}')

── Training Configuration ──
  model               : yolov8n.pt
  path                : data/processed/gtsdb
  train               : train/images
  val                 : val/images
  test                : test/images
  nc                  : 3
  names               : {0: 'prohibitory', 1: 'danger', 2: 'mandatory'}
  epochs              : 50
  imgsz               : 640
  batch               : 16
  optimizer           : AdamW
  lr0                 : 0.001
  lrf                 : 0.01
  momentum            : 0.937
  weight_decay        : 0.0005
  warmup_epochs       : 3
  patience            : 10
  hsv_h               : 0.015
  hsv_s               : 0.7
  hsv_v               : 0.4
  degrees             : 15.0
  translate           : 0.1
  scale               : 0.5
  shear               : 0.0
  flipud              : 0.0
  fliplr              : 0.0
  mosaic              : 1.0
  mixup               : 0.0
  project             : runs/train
  name                : gtsdb_yolov8n_v1
  save       

## 2. Run Training

In [3]:
# ── Training parameters (override config values here if needed) ────────
MODEL_WEIGHTS = 'yolov8n.pt'   # COCO pretrained weights
EPOCHS        = 50
IMG_SIZE      = 640
BATCH_SIZE    = 16             # Reduce to 8 if GPU runs out of memory
RUN_NAME      = 'gtsdb_yolov8n_v1'

print(f'Starting training: {MODEL_WEIGHTS} for {EPOCHS} epochs on device={DEVICE}')
print('This will take ~30–60 min on a GPU, several hours on CPU.')
print()

model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data    = str(CONFIG_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    name    = RUN_NAME,
    project = str(PROJECT_ROOT / 'runs' / 'train'),
    exist_ok= True,
    # Augmentation overrides (fliplr=0 is critical for direction-sensitive signs)
    fliplr  = 0.0,
    degrees = 15.0,
    hsv_v   = 0.4,
    hsv_s   = 0.7,
)

print('\n✅ Training complete!')
BEST_WEIGHTS = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'Best weights saved at: {BEST_WEIGHTS}')

Starting training: yolov8n.pt for 50 epochs on device=cpu
This will take ~30–60 min on a GPU, several hours on CPU.

New https://pypi.org/project/ultralytics/8.4.92 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.90  Python-3.14.3 torch-2.13.0+cpu CPU (12th Gen Intel Core i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\traffic-sign-detection\configs\gtsdb_yolov8n.yaml, degrees=15.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_widt

## 3. Plot Training Curves

In [4]:
run_dir = PROJECT_ROOT / 'runs' / 'train' / RUN_NAME
csv_path = run_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()  # remove leading spaces from column names
    print('Columns:', df.columns.tolist())

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f'Training Curves — {RUN_NAME}', fontsize=14)

    metrics_to_plot = [
        ('train/box_loss',  'Train Box Loss',  'red'),
        ('train/cls_loss',  'Train Cls Loss',  'orange'),
        ('val/box_loss',    'Val Box Loss',    'blue'),
        ('val/cls_loss',    'Val Cls Loss',    'purple'),
        ('metrics/mAP50',   'mAP@0.5',         'green'),
        ('metrics/mAP50-95','mAP@0.5:0.95',    'teal'),
    ]

    for ax, (col, label, colour) in zip(axes.flatten(), metrics_to_plot):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=colour, lw=2)
            ax.set_title(label, fontsize=11)
            ax.set_xlabel('Epoch')
            ax.grid(alpha=0.3)
        else:
            ax.set_title(f'{label} (not found)', fontsize=11)

    plt.tight_layout()
    out = PROJECT_ROOT / 'results' / 'figures' / f'training_curves_{RUN_NAME}.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print(f'Saved → {out}')
else:
    print(f'results.csv not found at {csv_path} — run training first.')

Columns: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


<Figure size 1600x800 with 6 Axes>

Saved → E:\traffic-sign-detection\results\figures\training_curves_gtsdb_yolov8n_v1.png


## 4. Quick Validation Check After Training

In [5]:
BEST_WEIGHTS = PROJECT_ROOT / 'runs' / 'train' / RUN_NAME / 'weights' / 'best.pt'

if BEST_WEIGHTS.exists():
    best_model = YOLO(str(BEST_WEIGHTS))
    val_results = best_model.val(
        data  = str(CONFIG_PATH),
        split = 'val',
        device= DEVICE,
    )
    print('\n── Validation Results (Best Checkpoint) ──')
    print(f'  mAP@0.5      : {val_results.box.map50:.4f}')
    print(f'  mAP@0.5:0.95 : {val_results.box.map:.4f}')
    print(f'  Precision    : {val_results.box.mp:.4f}')
    print(f'  Recall       : {val_results.box.mr:.4f}')
else:
    print(f'Best weights not found at {BEST_WEIGHTS}. Complete training first.')

Ultralytics 8.4.90  Python-3.14.3 torch-2.13.0+cpu CPU (12th Gen Intel Core i5-12450H)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 881.5340.9 MB/s, size: 336.0 KB)
val: Scanning E:\traffic-sign-detection\data\processed\gtsdb\val\labels.cache... 80 images, 11 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 21.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.1s/it 10.3s.7ss
                   all         80        110       0.85      0.752      0.852      0.662
           prohibitory         33         42      0.827      0.913      0.927      0.787
                danger         37         49      0.867      0.714      0.846      0.635
             mandatory         17         19      0.857      0.629      0.783      0.565
Speed: 1.2ms preprocess, 106.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to E:\traf

## 5. Ablation — With vs Without Augmentation

*Run this after the main training above. Trains a second model variant.*

In [ ]:
# ── Ablation: no augmentation ──────────────────────────────────────────
RUN_NAME_NO_AUG = 'gtsdb_yolov8n_noaug_v1'

model_no_aug = YOLO(MODEL_WEIGHTS)
results_no_aug = model_no_aug.train(
    data    = str(CONFIG_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    name    = RUN_NAME_NO_AUG,
    project = str(PROJECT_ROOT / 'runs' / 'train'),
    exist_ok= True,
    # Disable all augmentation
    fliplr  = 0.0, flipud = 0.0, degrees = 0.0,
    hsv_h   = 0.0, hsv_s  = 0.0, hsv_v   = 0.0,
    mosaic  = 0.0, mixup  = 0.0, scale   = 0.0,
    translate = 0.0,
)

print('Ablation (no augmentation) training complete.')

New https://pypi.org/project/ultralytics/8.4.92 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.90  Python-3.14.3 torch-2.13.0+cpu CPU (12th Gen Intel Core i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\traffic-sign-detection\configs\gtsdb_yolov8n.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0

## 6. Ablation — Model Size Comparison (nano vs small)

*Optional: trains YOLOv8s for the model-size ablation.*

In [ ]:
RUN_NAME_SMALL = 'gtsdb_yolov8s_v1'

model_small = YOLO('yolov8s.pt')   # Small variant (~11 MB vs ~6 MB nano)
results_small = model_small.train(
    data    = str(CONFIG_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = 8,          # smaller batch for larger model
    device  = DEVICE,
    name    = RUN_NAME_SMALL,
    project = str(PROJECT_ROOT / 'runs' / 'train'),
    exist_ok= True,
    fliplr  = 0.0,
    degrees = 15.0,
)

print('YOLOv8s training complete.')

## 7. Copy Best Checkpoint to results/

For easy access during evaluation and the demo.

In [ ]:
import shutil

checkpoints_dir = PROJECT_ROOT / 'results' / 'checkpoints'
checkpoints_dir.mkdir(parents=True, exist_ok=True)

for run_name in [RUN_NAME, RUN_NAME_NO_AUG, RUN_NAME_SMALL]:
    best = PROJECT_ROOT / 'runs' / 'train' / run_name / 'weights' / 'best.pt'
    if best.exists():
        dest = checkpoints_dir / f'{run_name}_best.pt'
        shutil.copy2(best, dest)
        print(f'Copied: {dest}')
    else:
        print(f'Not found (training may not be complete): {best}')

print('\n⚠️  Remember: do NOT commit .pt files to GitHub. Provide a download link instead.')